# Thesis: Entity-Aware A-RAG with Evidence Verification

**Đề tài**: Nghiên cứu cải tiến mô hình A-RAG dựa trên theo dõi thực thể và kiểm chứng bằng chứng trong hỏi đáp đa bước

**Branch**: `thesis-entity-evidence-arag`

**Thứ tự chạy**: Cell 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8 → 9 → 10 → 11

## Cell 1: Clone Repo / Pull Code

In [ ]:
import os, subprocess

REPO_URL = "https://github.com/trangdx2602/arag.git"
DATA_URL = "https://huggingface.co/datasets/Ayanami0730/rag_test"
REPO_DIR = "/content/arag"
BRANCH = "thesis-entity-evidence-arag"
FORCE_RECLONE = False  # Set True nếu muốn xóa và clone lại từ đầu

# --- Clone / pull repo ---
if FORCE_RECLONE and os.path.exists(REPO_DIR):
    import shutil; shutil.rmtree(REPO_DIR)
    print("Removed existing repo for re-clone.")

if os.path.exists(REPO_DIR):
    remote = subprocess.run(["git", "-C", REPO_DIR, "remote", "get-url", "origin"],
                            capture_output=True, text=True).stdout.strip()
    if "trangdx2602" not in remote:
        import shutil; shutil.rmtree(REPO_DIR)
        print(f"Wrong remote ({remote}), re-cloning from {REPO_URL}...")
        !git clone {REPO_URL} {REPO_DIR}
        !cd {REPO_DIR} && git checkout {BRANCH}
    else:
        print("Repo already exists — pulling latest...")
        !cd {REPO_DIR} && git fetch origin && git checkout {BRANCH} && git pull origin {BRANCH}
else:
    !git clone {REPO_URL} {REPO_DIR}
    !cd {REPO_DIR} && git checkout {BRANCH}

# --- Download dataset from HuggingFace ---
DATA_DIR = f"{REPO_DIR}/data"
if not os.path.exists(f"{DATA_DIR}/musique/chunks.json"):
    print("Downloading dataset from HuggingFace (2-5 phút)...")
    !pip install huggingface_hub -q
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id="Ayanami0730/rag_test",
        repo_type="dataset",
        local_dir=DATA_DIR,
        ignore_patterns=["*.git*"],
    )
    print("Dataset downloaded.")
else:
    print("Dataset already present.")

!ls {REPO_DIR}

# Change working directory so relative paths in YAML configs resolve correctly
os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## Cell 2: Cài Môi Trường + Set API Key

In [1]:
!pip install -e "/content/arag[full]" -q

import os

# ============================================================
# NHẬP API KEY CỦA BẠN TẠI ĐÂY
# (Nếu dùng Colab Secrets thì để trống — xem hướng dẫn bên dưới)
# ============================================================
ARAG_API_KEY_MANUAL = ""   # <-- Dán API key vào đây, VD: "sk-proj-abc123..."
ARAG_MODEL_MANUAL   = "gpt-4o-mini"   # <-- Model, VD: "gpt-4o-mini" (để trống = dùng default)
ARAG_BASE_URL_MANUAL = "https://api.openai.com/v1"  # <-- Base URL nếu không phải OpenAI (để trống = OpenAI)

if ARAG_API_KEY_MANUAL:
    os.environ["ARAG_API_KEY"]  = ARAG_API_KEY_MANUAL
    os.environ["ARAG_MODEL"]    = ARAG_MODEL_MANUAL or "gpt-4o-mini"
    os.environ["ARAG_BASE_URL"] = ARAG_BASE_URL_MANUAL or "https://api.openai.com/v1"
    print("API key set manually.")
else:
    # Thử load từ Colab Secrets (Sidebar → biểu tượng 🔑 → Add new secret)
    try:
        from google.colab import userdata
        key = userdata.get("ARAG_API_KEY")
        if not key:
            raise ValueError("Empty")
        os.environ["ARAG_API_KEY"]  = key
        os.environ["ARAG_MODEL"]    = userdata.get("ARAG_MODEL") or "gpt-4o-mini"
        os.environ["ARAG_BASE_URL"] = userdata.get("ARAG_BASE_URL") or "https://api.openai.com/v1"
        print("API key loaded from Colab Secrets.")
    except Exception:
        print("WARNING: ARAG_API_KEY chưa được set!")
        print("  → Cách 1: Điền ARAG_API_KEY_MANUAL ở trên rồi chạy lại cell này")
        print("  → Cách 2: Sidebar trái → biểu tượng 🔑 Secrets → Add new secret")

print("API key set  :", bool(os.environ.get("ARAG_API_KEY")))
print("Model        :", os.environ.get("ARAG_MODEL", "gpt-4o-mini"))
print("Base URL     :", os.environ.get("ARAG_BASE_URL", "https://api.openai.com/v1"))

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for arag (pyproject.toml) ... done
API key set manually.
API key set  : True
Model        : gpt-4o-mini
Base URL     : https://api.openai.com/v1


## Cell 3: Mount Google Drive

In [2]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE_RESULTS_DIR = "/content/drive/MyDrive/thesis_arag_results"
import os
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
print(f"Results will be saved to: {DRIVE_RESULTS_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Results will be saved to: /content/drive/MyDrive/thesis_arag_results


## Cell 4: Build Index HotpotQA / MuSiQue (GPU)

In [3]:
import os, gc
REPO_DIR = "/content/arag"

# Avoid CUDA OOM when encoding large sentence sets
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Build MuSiQue index (batch_size=8 to avoid OOM)
!python {REPO_DIR}/scripts/build_index.py --chunks {REPO_DIR}/data/musique/chunks.json --output {REPO_DIR}/data/musique/index --model Qwen/Qwen3-Embedding-0.6B --device cuda:0 --batch-size 8

# Clear VRAM before second index build
import torch
torch.cuda.empty_cache(); gc.collect()
print("VRAM cleared.")

# Build HotpotQA index
!python {REPO_DIR}/scripts/build_index.py --chunks {REPO_DIR}/data/hotpotqa/chunks.json --output {REPO_DIR}/data/hotpotqa/index --model Qwen/Qwen3-Embedding-0.6B --device cuda:0 --batch-size 8

print("Both indexes built.")

Loading chunks from: /content/arag/data/musique/chunks.json
Loaded 1354 chunks
Extracting sentences...
Processing chunks: 100% 1354/1354 [00:00<00:00, 9113.69it/s]
Total sentences: 50767
Loading model: Qwen/Qwen3-Embedding-0.6B
Loading weights: 100% 310/310 [00:00<00:00, 1032.44it/s, Materializing param=norm.weight]
/content/arag/scripts/build_index.py:80: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded. Embedding dimension: {model.get_sentence_embedding_dimension()}")
Model loaded. Embedding dimension: 1024
Encoding sentences...
Batches: 100% 6346/6346 [16:25<00:00,  6.44it/s]
Saving index to: /content/arag/data/musique/index/sentence_index.pkl
Index built successfully!
  - Chunks: 1354
  - Sentences: 50767
  - Embedding dim: 1024
VRAM cleared.
Loading chunks from: /content/arag/data/hotpotqa/chunks.json
Loaded 1311 chunks
Extracting sentences...
Processing chunks: 100% 1311/1311 [00:00<00:00, 9109.53it

## Cell 5: Chạy Naive RAG

In [4]:
import os
os.chdir("/content/arag")  # ensure CWD is repo root
REPO_DIR = "/content/arag"
LIMIT = 20
WORKERS = 5

# Pull latest code (get CWD fix if not done via Cell 1)
!git -C /content/arag pull origin thesis-entity-evidence-arag -q

!python {REPO_DIR}/scripts/thesis/run_variants.py --config {REPO_DIR}/configs/thesis/musique_base.yaml --variant naive_rag --questions {REPO_DIR}/data/musique/questions.json --output {REPO_DIR}/results/thesis/naive_rag_musique --limit {LIMIT} --workers {WORKERS}

print("Naive RAG done.")

Variant: naive_rag
Total: 20 | Completed: 0 | Pending: 20
Traceback (most recent call last):
  File "/content/arag/scripts/thesis/run_variants.py", line 272, in <module>
    main()
  File "/content/arag/scripts/thesis/run_variants.py", line 244, in main
    shared_tools = build_tools(config, use_entity_lookup=use_entity_lookup,
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/arag/scripts/thesis/run_variants.py", line 71, in build_tools
    tools.register(KeywordSearchTool(chunks_file=chunks_file))
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/arag/src/arag/tools/keyword_search.py", line 24, in __init__
    self.chunks = self._load_chunks()
                  ^^^^^^^^^^^^^^^^^^^
  File "/content/arag/src/arag/tools/keyword_search.py", line 31, in _load_chunks
    with open(self.chunks_file, 'r', encoding='utf-8') as f:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such f

## Cell 6: Chạy A-RAG Baseline

In [5]:
import os
os.chdir("/content/arag")  # ensure CWD is repo root
REPO_DIR = "/content/arag"
LIMIT = 20
WORKERS = 5

!python {REPO_DIR}/scripts/thesis/run_variants.py --config {REPO_DIR}/configs/thesis/musique_base.yaml --variant arag_baseline --questions {REPO_DIR}/data/musique/questions.json --output {REPO_DIR}/results/thesis/arag_baseline_musique --limit {LIMIT} --workers {WORKERS}

print("A-RAG Baseline done.")

Variant: arag_baseline
Total: 20 | Completed: 0 | Pending: 20
Traceback (most recent call last):
  File "/content/arag/scripts/thesis/run_variants.py", line 272, in <module>
    main()
  File "/content/arag/scripts/thesis/run_variants.py", line 244, in main
    shared_tools = build_tools(config, use_entity_lookup=use_entity_lookup,
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/arag/scripts/thesis/run_variants.py", line 71, in build_tools
    tools.register(KeywordSearchTool(chunks_file=chunks_file))
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/arag/src/arag/tools/keyword_search.py", line 24, in __init__
    self.chunks = self._load_chunks()
                  ^^^^^^^^^^^^^^^^^^^
  File "/content/arag/src/arag/tools/keyword_search.py", line 31, in _load_chunks
    with open(self.chunks_file, 'r', encoding='utf-8') as f:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No su

## Cell 7: Chạy A-RAG + Entity Tracker

In [6]:
import os
os.chdir("/content/arag")  # ensure CWD is repo root
REPO_DIR = "/content/arag"
LIMIT = 20
WORKERS = 5

!python {REPO_DIR}/scripts/thesis/run_variants.py --config {REPO_DIR}/configs/thesis/musique_entity.yaml --variant arag_entity_tracker --questions {REPO_DIR}/data/musique/questions.json --output {REPO_DIR}/results/thesis/arag_entity_tracker_musique --limit {LIMIT} --workers {WORKERS}

print("A-RAG + Entity Tracker done.")

Variant: arag_entity_tracker
Total: 20 | Completed: 0 | Pending: 20
Traceback (most recent call last):
  File "/content/arag/scripts/thesis/run_variants.py", line 272, in <module>
    main()
  File "/content/arag/scripts/thesis/run_variants.py", line 244, in main
    shared_tools = build_tools(config, use_entity_lookup=use_entity_lookup,
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/arag/scripts/thesis/run_variants.py", line 71, in build_tools
    tools.register(KeywordSearchTool(chunks_file=chunks_file))
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/arag/src/arag/tools/keyword_search.py", line 24, in __init__
    self.chunks = self._load_chunks()
                  ^^^^^^^^^^^^^^^^^^^
  File "/content/arag/src/arag/tools/keyword_search.py", line 31, in _load_chunks
    with open(self.chunks_file, 'r', encoding='utf-8') as f:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2]

## Cell 8: Chạy A-RAG + Evidence Checker

In [7]:
import os
os.chdir("/content/arag")  # ensure CWD is repo root
REPO_DIR = "/content/arag"
LIMIT = 20
WORKERS = 5

!python {REPO_DIR}/scripts/thesis/run_variants.py --config {REPO_DIR}/configs/thesis/musique_evidence.yaml --variant arag_evidence_checker --questions {REPO_DIR}/data/musique/questions.json --output {REPO_DIR}/results/thesis/arag_evidence_checker_musique --limit {LIMIT} --workers {WORKERS}

print("A-RAG + Evidence Checker done.")

Variant: arag_evidence_checker
Total: 20 | Completed: 0 | Pending: 20
Traceback (most recent call last):
  File "/content/arag/scripts/thesis/run_variants.py", line 272, in <module>
    main()
  File "/content/arag/scripts/thesis/run_variants.py", line 244, in main
    shared_tools = build_tools(config, use_entity_lookup=use_entity_lookup,
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/arag/scripts/thesis/run_variants.py", line 71, in build_tools
    tools.register(KeywordSearchTool(chunks_file=chunks_file))
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/arag/src/arag/tools/keyword_search.py", line 24, in __init__
    self.chunks = self._load_chunks()
                  ^^^^^^^^^^^^^^^^^^^
  File "/content/arag/src/arag/tools/keyword_search.py", line 31, in _load_chunks
    with open(self.chunks_file, 'r', encoding='utf-8') as f:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 

## Cell 9: Chạy A-RAG + Full (ET + EV)

In [8]:
import os
os.chdir("/content/arag")  # ensure CWD is repo root
REPO_DIR = "/content/arag"
LIMIT = 20
WORKERS = 5

!python {REPO_DIR}/scripts/thesis/run_variants.py --config {REPO_DIR}/configs/thesis/musique_full.yaml --variant arag_entity_evidence_full --questions {REPO_DIR}/data/musique/questions.json --output {REPO_DIR}/results/thesis/arag_entity_evidence_full_musique --limit {LIMIT} --workers {WORKERS}

print("A-RAG Full done.")

Variant: arag_entity_evidence_full
Total: 20 | Completed: 0 | Pending: 20
Traceback (most recent call last):
  File "/content/arag/scripts/thesis/run_variants.py", line 272, in <module>
    main()
  File "/content/arag/scripts/thesis/run_variants.py", line 244, in main
    shared_tools = build_tools(config, use_entity_lookup=use_entity_lookup,
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/arag/scripts/thesis/run_variants.py", line 71, in build_tools
    tools.register(KeywordSearchTool(chunks_file=chunks_file))
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/arag/src/arag/tools/keyword_search.py", line 24, in __init__
    self.chunks = self._load_chunks()
                  ^^^^^^^^^^^^^^^^^^^
  File "/content/arag/src/arag/tools/keyword_search.py", line 31, in _load_chunks
    with open(self.chunks_file, 'r', encoding='utf-8') as f:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Er

## Cell 10: Evaluate và Xuất Bảng So Sánh

In [9]:
import subprocess, json
REPO_DIR = "/content/arag"

# Quick contain-match comparison (no LLM needed)
!python {REPO_DIR}/scripts/thesis/compare_results.py \
    --results {REPO_DIR}/results/thesis/ \
    --dataset musique

# Optional: LLM-based accuracy evaluation (costs money)
# for variant in ["naive_rag", "arag_baseline", "arag_entity_tracker",
#                 "arag_evidence_checker", "arag_entity_evidence_full"]:
#     !python {REPO_DIR}/scripts/eval.py \
#         --predictions {REPO_DIR}/results/thesis/{variant}_musique/predictions.jsonl \
#         --config {REPO_DIR}/configs/thesis/musique_base.yaml \
#         --workers 5

# Display comparison JSON
import json
cmp_file = f"{REPO_DIR}/results/thesis/comparison_musique.json"
try:
    with open(cmp_file) as f:
        cmp = json.load(f)
    import pandas as pd
    rows = []
    for v, stats in cmp.items():
        if stats:
            rows.append({"variant": v, **stats})
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
except Exception as e:
    print(f"Cannot display table: {e}")


=== Thesis Results Comparison — MUSIQUE ===

  naive_rag: no results found at /content/arag/results/thesis/musique_naive_rag/predictions.jsonl
  arag_baseline: no results found at /content/arag/results/thesis/musique_arag_baseline/predictions.jsonl
  arag_entity_tracker: no results found at /content/arag/results/thesis/musique_arag_entity_tracker/predictions.jsonl
  arag_evidence_checker: no results found at /content/arag/results/thesis/musique_arag_evidence_checker/predictions.jsonl
  arag_entity_evidence_full: no results found at /content/arag/results/thesis/musique_arag_entity_evidence_full/predictions.jsonl

---------------------------------------------------------------------------------------------------------------------------
Variant                       N       Contain-Acc   Avg Loops    Avg Tokens    Avg Cost($)   Avg Entities    Avg Coverage  
---------------------------------------------------------------------------------------------------------------------------
naive_r

## Cell 11: Copy Results về Google Drive

In [10]:
import shutil, os
REPO_DIR = "/content/arag"
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/thesis_arag_results"

src = f"{REPO_DIR}/results/thesis"
dst = f"{DRIVE_RESULTS_DIR}/results_thesis"

if os.path.exists(src):
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f"Results copied to: {dst}")
else:
    print("No results to copy yet.")

# List saved files
for root, dirs, files in os.walk(dst):
    for f in files:
        path = os.path.join(root, f)
        size = os.path.getsize(path)
        print(f"  {path.replace(dst, '')} ({size:,} bytes)")

Results copied to: /content/drive/MyDrive/thesis_arag_results/results_thesis
  /comparison_musique.json (149 bytes)
